In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
import os
import re

/home/lu.dong1/.conda/envs/expo-judge/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_path = "/projects/insightx-lab/cn_grpo/models/Llama-3.1-8B-Instruct"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:23<00:00, 12.46it/s, Materializing param=model.norm.weight]                              


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [9]:
def build_prompt(raw_text):
    return f"""
Extract the following four fields from the receipt text:

- company
- date
- address
- total

Return ONLY a valid JSON object.
Do not include explanation.
Do not include extra text.
Do not repeat the input.

If a field is missing, return null.

Receipt Text:
{raw_text}
"""

In [10]:
raw_text_path = "data/temp/X00016469612.txt"

In [11]:
with open(raw_text_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [1]:
messages = [
    {"role": "system", "content": "You are a precise information extraction system."},
    {"role": "user", "content": prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.0,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

# 只取 assistant 输出
generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

NameError: name 'prompt' is not defined

In [16]:
print(response)

{
  "company": "BOOK TA-K (TAMAN DAYA) SDN BHD",
  "date": "25/12/2018 8:13:39 PM",
  "address": "NO.5? 55,57 & 59, JALAN SAGU 18, TAMAN DAYA 81100 JOHOR BAHRU, JOHOR.",
  "total": "9.60"
}


In [18]:
def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group()
    return None

json_str = extract_json(response)

if json_str is None:
    raise ValueError("No valid JSON found in response")

extracted_data = json.loads(json_str)

In [20]:
engine_output = {
    "engine": "llama3.1-8B-Instruct",
    "source_file": raw_text_path,
    "extracted_fields": extracted_data
}

In [21]:
output_path = "data/temp/X00016469612_engineA.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(engine_output, f, indent=4, ensure_ascii=False)

# Judgement Agent

## Module-1

In [2]:
import json
import re
from dateutil import parser

In [3]:
raw_text_path = "data/temp/X00016469612.txt"
engineA_path  = "data/temp/X00016469612_engineA.json"

In [43]:
output_path = "data/temp/X00016469612_module1.json"

### Raw text normalization utils

In [5]:
def normalize_line(s):
    if s is None:
        return ""
        
    s = s.replace("\x0c", " ")           # form feed
    s = s.replace("\u00a0", " ")         # non-breaking space
    s = re.sub(r"[ \t]+", " ", s)        # collapse spaces/tabs
    
    return s.strip()


In [6]:
def tokenize_text(s):
    if not s:
        return []
        
    return re.findall(r"[A-Za-z]+|\d+(?:\.\d+)?|[^\sA-Za-z0-9]", s)

In [42]:
import math
import re
from typing import Dict, List, Optional

# Table/item header cues (SROIE receipts/invoices)
ITEMS_ANCHORS = [
    r"\bDESC\b", r"\bITEM\b", r"\bQTY\b", r"\bPRICE\b", r"\bAMOUNT\b",
    r"\bCODE\b", r"\bU\.?\s*PRICE\b", r"\bUNIT\b.*\bPRICE\b", r"\bS\/PRICE\b",
    r"\bTOTAL\s*QTY\b"
]

SEPARATOR_RE = r"^[-=]{3,}$"


def find_items_anchor(lines_norm: List[str], start: int, end: int) -> Optional[int]:
    """
    Returns the first line index in [start, end) that looks like an item/table header.
    """
    end = min(end, len(lines_norm))
    for i in range(start, end):
        u = lines_norm[i].upper().strip()
        if not u:
            continue
        # separator line often appears right before/after table header
        if re.search(SEPARATOR_RE, u):
            return i
        if any(re.search(p, u) for p in ITEMS_ANCHORS):
            return i
    return None

# end-exclusive
def regions(
    lines_norm: List[str],
    header_ratio: float = 0.25,
    min_lines: int = 8,
    max_lines: int = 30,
    lookahead: int = 10
) -> Dict[str, List[int]]:
    """
    Hybrid header range inference:
      1) Compute a stable baseline header_end using a top-ratio heuristic.
      2) Search for an items/table anchor within [0, header_end_base + lookahead).
         If found, cut header_end before that anchor (soft cut).
    Output:
      {"header_line_range": [0, header_end]}
    """
    n = len(lines_norm)
    if n == 0:
        return {"header_line_range": [0, 0]}

    # 1) Ratio-based baseline
    header_end_base = int(math.ceil(n * header_ratio))
    header_end_base = max(header_end_base, min_lines)
    header_end_base = min(header_end_base, max_lines, n)

    # 2) Soft cut using a wider search window
    search_end = min(n, header_end_base + lookahead)
    anchor_idx = find_items_anchor(lines_norm, 0, search_end)

    if anchor_idx is not None and anchor_idx >= 3:
        header_end = min(anchor_idx, header_end_base) if anchor_idx <= header_end_base + lookahead else header_end_base
        # If anchor appears slightly after base, it's still likely header ends before it
        header_end = anchor_idx
    else:
        header_end = header_end_base

    # Safety clamp
    header_end = max(min_lines, min(header_end, n))

    return {"header_line_range": [0, header_end]}

### Field normalization utils

In [8]:
def norm_company(x):
    if x is None:
        return None
    x = x.strip()
    return x if x else None

In [9]:
def norm_address(x):
    if x is None:
        return None
    x = normalize_line(x)
    x = x.rstrip(".")
    return x if x else None

In [10]:
def norm_date(x):
    """
    norm_value: ISO date string YYYY-MM-DD
    parsed: {year, month, day}
    """
    if x is None:
        return None, None
    x = x.strip()
    if not x:
        return None, None
    try:
        dt = parser.parse(x, dayfirst=True, fuzzy=True)
        return dt.strftime("%Y-%m-%d"), {"year": dt.year, "month": dt.month, "day": dt.day}
    except Exception:
        return None, None

In [11]:
def norm_total(x):
    """
    norm_value: float or None
    parsed: {amount, currency}
    """
    if x is None:
        return None, None
    x = x.strip()
    if not x:
        return None, None
    try:
        amt = float(x.replace(",", ""))
        return amt, {"amount": amt, "currency": None}
    except Exception:
        return None, None

### Build raw_text

In [36]:
with open(raw_text_path, "r", encoding="utf-8") as f:
    full_raw = f.read()

lines_raw = full_raw.splitlines()
lines_norm = [normalize_line(l) for l in lines_raw]
tokens_per_line = [tokenize_text(l) for l in lines_norm]
regions = regions(lines_norm)

raw_text_block = {
    "full_raw": full_raw,
    "lines_raw": lines_raw,
    "lines_norm": lines_norm,
    "tokens_per_line": tokens_per_line,
    "regions": regions
}


### Build engine fields block

In [37]:
with open(engineA_path, "r", encoding="utf-8") as f:
    engineA_data = json.load(f)

extracted = engineA_data.get("extracted_fields", {})

In [38]:
company_raw = extracted.get("company")
address_raw = extracted.get("address")
date_raw    = extracted.get("date")
total_raw   = extracted.get("total")

company_norm = norm_company(company_raw)
address_norm = norm_address(address_raw)
date_norm, date_parsed = norm_date(date_raw)
total_norm, total_parsed = norm_total(total_raw)

In [39]:
engineA_block = {
    "company": {
        "raw_value": company_raw if company_raw is not None else None,
        "norm_value": company_norm,
        "tokens": tokenize_text(company_norm) if company_norm else []
    },
    "date": {
        "raw_value": date_raw if date_raw is not None else None,
        "norm_value": date_norm,
        "tokens": tokenize_text(date_norm) if date_norm else [],
        "parsed": date_parsed
    },
    "address": {
        "raw_value": address_raw if address_raw is not None else None,
        "norm_value": address_norm,
        "tokens": tokenize_text(address_norm) if address_norm else []
    },
    "total": {
        "raw_value": total_raw if total_raw is not None else None,
        "norm_value": total_norm,
        "tokens": tokenize_text(str(total_norm)) if total_norm is not None else [],
        "parsed": total_parsed
    }
}

### Assemble output

In [40]:
module1_output = {
    "source_file": raw_text_path,
    "raw_text": raw_text_block,
    "engines": {
        "engineA": engineA_block
    }
}

print(module1_output)

{'source_file': 'data/temp/X00016469612.txt', 'raw_text': {'full_raw': 'tan woon yann\n\nBOOK TA-K (TAMAN DAYA) SDN BHD\nB94 7-W\nNO.5? 55,57 & 59, JALAN SAGU 18,\nTAMAN DAYA\n81100 JOHOR BAHRU,\nJOHOR.\n\nLAM MCU\n\nDocument Ho : TDO1167104\n\n \n\nDate 25/12/2018 8:13:39 PM\nCashier MANIS\nMember\nCASH BILL\nCODE/DESC PRICE — Disc AMOUIT\nQuy RM RM\n9556939040118 KF MODELLING CLAY KIDDY FISH\n1PC * 9.00) 6,00 9.00\nTotal : 9.00\nRour ding Adjustment 0.00\n\nRound: :d Total (RM):\n\n9.60\n\nCash\nCHANGE\n\n  \n\nGOODS SOLD ARE NOT RETURNAR\nEXCHANGEABLE\n\n \n\nTHANK YOU.\nPLEASE COME AGATY t\n\x0c', 'lines_raw': ['tan woon yann', '', 'BOOK TA-K (TAMAN DAYA) SDN BHD', 'B94 7-W', 'NO.5? 55,57 & 59, JALAN SAGU 18,', 'TAMAN DAYA', '81100 JOHOR BAHRU,', 'JOHOR.', '', 'LAM MCU', '', 'Document Ho : TDO1167104', '', ' ', '', 'Date 25/12/2018 8:13:39 PM', 'Cashier MANIS', 'Member', 'CASH BILL', 'CODE/DESC PRICE — Disc AMOUIT', 'Quy RM RM', '9556939040118 KF MODELLING CLAY KIDDY FISH', '1PC * 

In [41]:
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(module1_output, f, indent=2, ensure_ascii=False)

## Module-2:Evidence Retrieval

In [1]:
import json
import re

In [2]:
input_path = "data/temp/X00016469612_module1.json"

In [3]:
with open(input_path, "r", encoding="utf-8") as f:
    doc = json.load(f)

In [4]:
# raw text
lines_norm = doc["raw_text"]["lines_norm"]
tokens_per_line = doc["raw_text"]["tokens_per_line"]
header_start, header_end = doc["raw_text"]["regions"]["header_line_range"]  # end-exclusive

header_line_idxs = list(range(header_start, header_end))
all_line_idxs = list(range(len(lines_norm)))

In [5]:
# engine fields
engine_name = "engineA"
engine = doc["engines"][engine_name]

fields = {
    "company": {
        "raw_value": engine["company"].get("raw_value"),
        "norm_value": engine["company"].get("norm_value"),
        "tokens": engine["company"].get("tokens", []),
    },
    "date": {
        "raw_value": engine["date"].get("raw_value"),
        "norm_value": engine["date"].get("norm_value"),
        "tokens": engine["date"].get("tokens", []),
        "parsed": engine["date"].get("parsed"),
    },
    "address": {
        "raw_value": engine["address"].get("raw_value"),
        "norm_value": engine["address"].get("norm_value"),
        "tokens": engine["address"].get("tokens", []),
    },
    "total": {
        "raw_value": engine["total"].get("raw_value"),
        "norm_value": engine["total"].get("norm_value"),
        "tokens": engine["total"].get("tokens", []),
        "parsed": engine["total"].get("parsed"),
    },
}

In [6]:
print("Loaded:", input_path)
print("source_file:", doc["source_file"])
print("num_lines:", len(lines_norm))
print("header_range:", (header_start, header_end))
print("header_last_line:", header_end - 1, lines_norm[header_end - 1] if header_end > 0 else None)
print()

Loaded: data/temp/X00016469612_module1.json
source_file: data/temp/X00016469612.txt
num_lines: 43
header_range: (0, 19)
header_last_line: 18 CASH BILL



In [7]:
for k, v in fields.items():
    print(k, "norm_value:", v["norm_value"], "tokens_len:", len(v["tokens"]))

company norm_value: BOOK TA-K (TAMAN DAYA) SDN BHD tokens_len: 10
date norm_value: 2018-12-25 tokens_len: 5
address norm_value: NO.5? 55,57 & 59, JALAN SAGU 18, TAMAN DAYA 81100 JOHOR BAHRU, JOHOR tokens_len: 21
total norm_value: 9.6 tokens_len: 1


In [8]:
def score_line(field_tokens, line_tokens):
    fset, lset = set(field_tokens), set(line_tokens)
    if not fset or not lset:
        return 0.0, 0.0, 0.0  # match, coverage, support
    
    inter = len(fset & lset)
    if inter == 0:
        return 0.0, 0.0, 0.0
    
    # coverage: how much of the field tokens appear in the line
    coverage = inter / len(fset)
    # specificity: how much of the line tokens are relevant to the field
    specificity = inter / len(lset)
    # match: F1 of (coverage, specificity)
    match = (2 * coverage * specificity / (coverage + specificity)) if (coverage + specificity) else 0.0
    
    # for now, support == coverage (no header prior)
    support = coverage
    return match, coverage, support


def find_best_and_candidates(field_tokens, tokens_per_line, candidate_idxs, topk=5, min_match=0.0):
    scored = []
    for i in candidate_idxs:
        match, cov, sup = score_line(field_tokens, tokens_per_line[i])
        if match > min_match:
            scored.append({
                "line_indices": [i],
                "match_score": match,
                "coverage_score": cov,
                "support_score": sup,
            })
    scored.sort(key=lambda x: x["support_score"], reverse=True)
    
    best = scored[0] if scored else None
    # candidates exclude best
    candidates = scored[1:topk] if len(scored) > 1 else []
    return best, candidates

### candidates for total (mandatory) + date, address, company

In [9]:
THRESH = 0.7  # trigger threshold for non-total fields

def should_trigger_alternatives(field_name, best):
    if field_name == "total":
        return True
    if best is None:
        return True
    return (best["support_score"] < THRESH) or (best["coverage_score"] < THRESH)

In [10]:
TOTAL_KEYWORDS = re.compile(r"\b(TOTAL|GRAND|ROUND|SUB\s*TOTAL|SUBTOTAL|AMOUNT)\b", re.IGNORECASE)

def extract_total_alternatives(lines_norm, topk=10):
    cands = []
    for i, line in enumerate(lines_norm):
        amounts = re.findall(r"\d+\.\d{2}", line)
        if not amounts:
            continue

        tag = "amount_only"
        if TOTAL_KEYWORDS.search(line):
            u = line.upper()
            if "GRAND" in u:
                tag = "grand_total_context"
            elif "ROUND" in u:
                tag = "round_total_context"
            elif "SUB" in u:
                tag = "sub_total_context"
            elif "TOTAL" in u:
                tag = "total_context"
            else:
                tag = "keyword_context"

        for a in amounts:
            cands.append({
                "amount_str": a,
                "amount": float(a),
                "line_indices": [i],
                "text": [line],
                "tag": tag,
            })

    def rank(x):
        kw_bonus = 1 if x["tag"] != "amount_only" else 0
        return (kw_bonus, x["amount"])

    cands.sort(key=rank, reverse=True)
    return cands[:topk]

In [11]:
DATE_PAT = re.compile(r"(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})")
DATE_KEYWORDS = re.compile(r"\b(DATE|TIME|PRINT)\b", re.IGNORECASE)

def extract_date_alternatives(lines_norm, topk=5):
    cands = []
    for i, line in enumerate(lines_norm):
        m = DATE_PAT.search(line)
        if not m:
            continue
        tag = "date_only"
        if DATE_KEYWORDS.search(line):
            tag = "date_keyword_context"
        cands.append({
            "value": m.group(1),
            "line_indices": [i],
            "text": [line],
            "tag": tag,
        })

    cands.sort(key=lambda x: 1 if x["tag"] == "date_keyword_context" else 0, reverse=True)
    return cands[:topk]

In [12]:
COMPANY_HINTS = re.compile(r"\b(SDN\s*BHD|LTD|INC|LLC|CO\.?|COMPANY|STORE|SHOP|ENTERPRISE)\b", re.IGNORECASE)

def extract_company_alternatives(field_tokens, tokens_per_line, lines_norm, best_line_idx=None, topk=5):
    scored = []
    for i, line in enumerate(lines_norm):
        match, cov, sup = score_line(field_tokens, tokens_per_line[i])
        if match <= 0:
            continue

        boost = 0.0
        u = line.upper().strip()
        if COMPANY_HINTS.search(u):
            boost += 0.05
        if len(u) <= 60 and len(re.findall(r"[A-Z]", u)) >= 6:
            boost += 0.03

        scored.append((sup + boost, match, cov, i))

    scored.sort(reverse=True)

    cands = []
    for sup, match, cov, i in scored:
        if best_line_idx is not None and i == best_line_idx:
            continue
        cands.append({
            "line_indices": [i],
            "match_score": match,
            "coverage_score": cov,
            "support_score": sup,
            "text": [lines_norm[i]],
        })
        if len(cands) >= topk:
            break
    return cands

In [13]:
ADDRESS_HINTS = re.compile(r"\b(NO\.?|JALAN|ROAD|STREET|AVE|BLVD|TAMAN|\d{5})\b", re.IGNORECASE)

def extract_address_alternatives(field_tokens, tokens_per_line, lines_norm, best_line_idx=None, topk=5):
    scored = []
    for i, line in enumerate(lines_norm):
        match, cov, sup = score_line(field_tokens, tokens_per_line[i])
        if match <= 0:
            continue

        boost = 0.0
        u = line.upper().strip()
        if ADDRESS_HINTS.search(u):
            boost += 0.05
        if len(re.findall(r"\d", u)) >= 3:
            boost += 0.02

        scored.append((sup + boost, match, cov, i))

    scored.sort(reverse=True)

    cands = []
    for sup, match, cov, i in scored:
        if best_line_idx is not None and i == best_line_idx:
            continue
        cands.append({
            "line_indices": [i],
            "match_score": match,
            "coverage_score": cov,
            "support_score": sup,
            "text": [lines_norm[i]],
        })
        if len(cands) >= topk:
            break
    return cands

In [14]:
# # Build module2 evidence (best/candidates + conditional alternatives)

# TOPK = 2

# for field_name in ["company", "date", "address", "total"]:
#     field_tokens = fields[field_name]["tokens"]

#     # total formatting assist: 9.6 -> 9.60
#     if field_name == "total":
#         nv = fields[field_name]["norm_value"]
#         if isinstance(nv, (int, float)):
#             field_tokens = list(field_tokens) + [f"{nv:.2f}"]

#     best, candidates = find_best_and_candidates(
#         field_tokens=field_tokens,
#         tokens_per_line=tokens_per_line,
#         candidate_idxs=all_line_idxs,
#         topk=TOPK + 1,  # +1 because candidates exclude best
#         min_match=0.0
#     )

#     evidence = {
#         "best": best,
#         "candidates": candidates,
#         "flags": {"triggered_alternatives": False}
#     }

#     if should_trigger_alternatives(field_name, best):
#         best_idx = best["line_indices"][0] if best else None

#         if field_name == "total":
#             evidence["alternatives"] = extract_total_alternatives(lines_norm, topk=TOPK)
#         elif field_name == "date":
#             evidence["alternatives"] = extract_date_alternatives(lines_norm, topk=TOPK)
#         elif field_name == "company":
#             evidence["alternatives"] = extract_company_alternatives(field_tokens, tokens_per_line, lines_norm, best_line_idx=best_idx, topk=TOPK)
#         elif field_name == "address":
#             evidence["alternatives"] = extract_address_alternatives(field_tokens, tokens_per_line, lines_norm, best_line_idx=best_idx, topk=TOPK)

#         evidence["flags"] = {
#             "triggered_alternatives": True,
#             "reason": "always_for_total" if field_name == "total" else "low_support_or_coverage"
#         }

#     doc["engines"][engine_name][field_name]["evidence"] = evidence

In [20]:
# Build module2 evidence (best/candidates + conditional alternatives)

TOPK = 2

def best_by_raw_substring(raw_value, lines_norm, candidate_idxs):
    if not raw_value:
        return None
    for i in candidate_idxs:
        if raw_value in lines_norm[i]:
            return {
                "line_indices": [i],
                "match_score": 1.0,
                "coverage_score": 1.0,
                "support_score": 1.0,
                "text": [lines_norm[i]],
            }
    return None

for field_name in ["company", "date", "address", "total"]:

    # --- date/total: raw_value exact substring match ---
    if field_name in ["date", "total"]:
        raw = fields[field_name]["raw_value"]
        best = best_by_raw_substring(raw, lines_norm, all_line_idxs)
        candidates = []  # keep it empty for now

        # if somehow not found, fallback to token overlap (optional)
        if best is None:
            field_tokens = fields[field_name]["tokens"]
            best, candidates = find_best_and_candidates(
                field_tokens=field_tokens,
                tokens_per_line=tokens_per_line,
                candidate_idxs=all_line_idxs,
                topk=TOPK + 1,
                min_match=0.0
            )

    # --- company/address: token overlap on norm tokens ---
    else:
        field_tokens = fields[field_name]["tokens"]
        best, candidates = find_best_and_candidates(
            field_tokens=field_tokens,
            tokens_per_line=tokens_per_line,
            candidate_idxs=all_line_idxs,
            topk=TOPK + 1,
            min_match=0.0
        )

    evidence = {
        "best": best,
        "candidates": candidates,
        "flags": {"triggered_alternatives": False}
    }

    # keep your alternatives logic as-is (total always, others conditional)
    if should_trigger_alternatives(field_name, best):
        best_idx = best["line_indices"][0] if best else None

        if field_name == "total":
            evidence["alternatives"] = extract_total_alternatives(lines_norm, topk=TOPK)
        elif field_name == "date":
            evidence["alternatives"] = extract_date_alternatives(lines_norm, topk=TOPK)
        elif field_name == "company":
            evidence["alternatives"] = extract_company_alternatives(field_tokens, tokens_per_line, lines_norm, best_line_idx=best_idx, topk=TOPK)
        elif field_name == "address":
            evidence["alternatives"] = extract_address_alternatives(field_tokens, tokens_per_line, lines_norm, best_line_idx=best_idx, topk=TOPK)

        evidence["flags"] = {
            "triggered_alternatives": True,
            "reason": "always_for_total" if field_name == "total" else "low_support_or_coverage"
        }

    doc["engines"][engine_name][field_name]["evidence"] = evidence

In [18]:
output_path = "data/temp/X00016469612_module2.json"

In [19]:
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(doc, f, indent=2, ensure_ascii=False)

## Module-3 Validity & Judgment Rules

In [1]:
import json
import math

In [2]:
input_path = "data/temp/X00016469612_module2.json"

In [3]:
with open(input_path, "r", encoding="utf-8") as f:
    doc = json.load(f)

In [4]:
engine_name = "engineA"
engine = doc["engines"][engine_name]

In [5]:
FIELDS = ["company", "date", "address", "total"]

In [6]:
# 3 states: pass, review_needed, fail

In [7]:
def worse_state(a, b):
    order = {"pass": 0, "review_needed": 1, "fail": 2}
    return a if order[a] >= order[b] else b

def has_question_mark(x):
    return isinstance(x, str) and ("?" in x)

def get_best(field_obj):
    return field_obj.get("evidence", {}).get("best")

def get_alternatives(field_obj):
    return field_obj.get("evidence", {}).get("alternatives")

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

In [8]:
# general rules
# 1. best evidence should exist
# 2. if contains question mark, then needs human reviews

def apply_general_rules(field_name, field_obj):
    state = "pass"
    reasons = []
    flags = {}

    best = get_best(field_obj)
    if best is None:
        return "fail", ["missing_best_evidence"], {"best_exists": False}

    flags["best_exists"] = True
    flags["best_line_indices"] = best.get("line_indices")
    flags["best_support_score"] = best.get("support_score")

    if has_question_mark(field_obj.get("raw_value")) or has_question_mark(field_obj.get("norm_value")):
        state = "review_needed"
        reasons.append("contains_question_mark")

    return state, reasons, flags

### field-specific rules

In [9]:
def rule_company(field_obj):
    state = "pass"
    reasons = []
    norm = field_obj.get("norm_value")
    toks = field_obj.get("tokens", [])

    if norm is None or str(norm).strip() == "":
        state = "fail"
        reasons.append("empty_company")
        return state, reasons

    if len(toks) < 2:
        state = "review_needed"
        reasons.append("company_name_too_short")

    return state, reasons

In [10]:
def rule_address(field_obj):
    state = "pass"
    reasons = []
    norm = field_obj.get("norm_value") or ""
    toks = field_obj.get("tokens", [])

    if str(norm).strip() == "":
        state = "fail"
        reasons.append("empty_address")
        return state, reasons

    # light sanity: address usually has digits or location cues
    s = str(norm).upper()
    has_digit = any(ch.isdigit() for ch in s)
    has_cue = any(k in s for k in ["NO", "JALAN", "ROAD", "STREET", "TAMAN", "AVE", "BLVD"])
    has_postcode = any(t.isdigit() and len(t) == 5 for t in toks)

    if not (has_digit or has_cue or has_postcode):
        state = "review_needed"
        reasons.append("address_missing_location_cues")

    return state, reasons

In [11]:
def rule_date(field_obj, year_min=1990, year_max=2050):
    state = "pass"
    reasons = []
    parsed = field_obj.get("parsed")

    if parsed is None:
        return "fail", ["date_parse_missing"]

    y = parsed.get("year")
    m = parsed.get("month")
    d = parsed.get("day")

    if not isinstance(y, int) or not isinstance(m, int) or not isinstance(d, int):
        return "fail", ["date_parse_invalid_types"]

    if not (year_min <= y <= year_max):
        return "fail", [f"date_year_out_of_range:{y}"]

    if not (1 <= m <= 12):
        return "fail", [f"date_month_out_of_range:{m}"]

    if not (1 <= d <= 31):
        return "fail", [f"date_day_out_of_range:{d}"]

    return state, reasons

In [12]:
def rule_total(field_obj):
    state = "pass"
    reasons = []
    flags = {}

    parsed = field_obj.get("parsed")
    if parsed is None or parsed.get("amount") is None:
        return "fail", ["total_parse_missing"], flags

    extracted_amount = safe_float(parsed.get("amount"))
    if extracted_amount is None:
        return "fail", ["total_amount_not_numeric"], flags

    # alternatives are expected for total
    alts = get_alternatives(field_obj) or []
    flags["num_alternatives"] = len(alts)

    # If no alternatives present, don't fail—just proceed
    if len(alts) == 0:
        return state, reasons, flags

    alt_amounts = []
    tagged_total_context = []
    for a in alts:
        amt = a.get("amount")
        if isinstance(amt, (int, float)):
            alt_amounts.append(float(amt))
        if a.get("tag") == "total_context" and isinstance(amt, (int, float)):
            tagged_total_context.append(float(amt))

    uniq = sorted(set(alt_amounts))
    flags["unique_alt_amounts"] = uniq

    # Hallucination check: extracted amount not present in any alt amounts
    if extracted_amount not in set(alt_amounts):
        return "fail", [f"total_not_supported_by_text:{extracted_amount}"], flags

    # Conflict check: multiple unique amounts in text
    if len(uniq) >= 2:
        state = "review_needed"
        reasons.append(f"total_conflict_candidates:{uniq}")

        # Optional: if there's a clear 'Total :' candidate and it differs, note it
        if len(tagged_total_context) > 0:
            # pick the first total_context amount (usually the right one)
            suggested = tagged_total_context[0]
            if suggested != extracted_amount:
                reasons.append(f"total_keyword_mismatch:extracted={extracted_amount},total_context={suggested}")
                flags["suggested_total_from_total_context"] = suggested

    return state, reasons, flags

In [13]:
field_results = {}
doc_state = "pass"
doc_flags = []

for field_name in FIELDS:
    fobj = engine[field_name]

    # general
    state, reasons, flags = apply_general_rules(field_name, fobj)

    # if general already fail, skip field rules
    if state != "fail":
        if field_name == "company":
            s2, r2 = rule_company(fobj)
            state = worse_state(state, s2)
            reasons += r2
        elif field_name == "address":
            s2, r2 = rule_address(fobj)
            state = worse_state(state, s2)
            reasons += r2
        elif field_name == "date":
            s2, r2 = rule_date(fobj)
            state = worse_state(state, s2)
            reasons += r2
        elif field_name == "total":
            s2, r2, f2 = rule_total(fobj)
            state = worse_state(state, s2)
            reasons += r2
            flags.update(f2)

    # doc-level aggregation
    doc_state = worse_state(doc_state, state)
    if field_name == "total" and any("total_conflict_candidates" in x for x in reasons):
        doc_flags.append("total_conflict")
    if any("contains_question_mark" in x for x in reasons):
        doc_flags.append(f"{field_name}_uncertain_ocr")

    field_results[field_name] = {
        "state": state,
        "reasons": reasons,
        "flags": flags,
        "evidence": {
            "best_line_indices": flags.get("best_line_indices")
        }
    }

In [14]:
module3_result = {
    "source_file": doc["source_file"],
    "engine": engine_name,
    "doc_state": doc_state,
    "doc_flags": sorted(set(doc_flags)),
    "field_results": field_results
}

print("doc_state:", module3_result["doc_state"])
print("doc_flags:", module3_result["doc_flags"])
print(json.dumps(module3_result["field_results"], indent=2, ensure_ascii=False))

doc_state: review_needed
doc_flags: ['address_uncertain_ocr', 'total_conflict']
{
  "company": {
    "state": "pass",
    "reasons": [],
    "flags": {
      "best_exists": true,
      "best_line_indices": [
        2
      ],
      "best_support_score": 1.0
    },
    "evidence": {
      "best_line_indices": [
        2
      ]
    }
  },
  "date": {
    "state": "pass",
    "reasons": [],
    "flags": {
      "best_exists": true,
      "best_line_indices": [
        15
      ],
      "best_support_score": 1.0
    },
    "evidence": {
      "best_line_indices": [
        15
      ]
    }
  },
  "address": {
    "state": "review_needed",
    "reasons": [
      "contains_question_mark"
    ],
    "flags": {
      "best_exists": true,
      "best_line_indices": [
        4
      ],
      "best_support_score": 0.7058823529411765
    },
    "evidence": {
      "best_line_indices": [
        4
      ]
    }
  },
  "total": {
    "state": "review_needed",
    "reasons": [
      "total_confli

In [15]:
output_path = "data/temp/X00016469612_module3.json"

In [16]:
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(module3_result, f, indent=2, ensure_ascii=False)